## timm

[PyTorch Image Models](https://timm.fast.ai/)（timm）是由 Ross Wightman 开发的一个出色的库，提供最先进的预训练计算机视觉模型。它类似于 Huggingface Transformers，但专注于计算机视觉而非 NLP（而且并不局限于基于 Transformer 的模型）！

Ross 非常慷慨地帮助我了解如何充分利用这个库，包括如何筛选出最优秀的模型。我将在这里分享我从他那里学到的内容，以及一些额外的思考。


## 数据

Ross 会定期对新加入 timm 的模型进行基准测试，并将结果保存为项目 GitHub 仓库中的 CSV 文件。为了分析这些数据，我们先来克隆该仓库：


In [ ]:
! git clone --depth 1 https://github.com/rwightman/pytorch-image-models.git
%cd pytorch-image-models/results

In [ ]:
import pandas as pd
df_results = pd.read_csv('results-imagenet.csv')

In [ ]:
def get_data(part, col):
    df = pd.read_csv(f'benchmark-{part}-amp-nhwc-pt111-cu113-rtx3090.csv').merge(df_results, on='model')
    df['secs'] = 1. / df[col]
    df['family'] = df.model.str.extract('^([a-z]+?(?:v2)?)(?:\d|_|$)')
    df = df[~df.model.str.endswith('gn')]
    df.loc[df.model.str.contains('in22'),'family'] = df.loc[df.model.str.contains('in22'),'family'] + '_in22'
    df.loc[df.model.str.contains('resnet.*d'),'family'] = df.loc[df.model.str.contains('resnet.*d'),'family'] + 'd'
    return df[df.family.str.contains('^re[sg]netd?|beit|convnext|levit|efficient|vit|vgg')]

In [ ]:
df = get_data('infer', 'infer_samples_per_sec')

## 推理结果


In [ ]:
import plotly.express as px
w,h = 1000,800

def show_all(df, title, size):
    return px.scatter(df, width=w, height=h, size=df[size]**2, title=title,
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
show_all(df, 'Inference', 'infer_img_size')

In [ ]:
subs = 'levit|resnetd?|regnetx|vgg|convnext.*|efficientnetv2|beit'

In [ ]:
def show_subs(df, title, size):
    df_subs = df[df.family.str.fullmatch(subs)]
    return px.scatter(df_subs, width=w, height=h, size=df_subs[size]**2, title=title,
        trendline="ols", trendline_options={'log_x':True},
        x='secs',  y='top1', log_x=True, color='family', hover_name='model', hover_data=[size])

In [ ]:
show_subs(df, 'Inference', 'infer_img_size')

In [ ]:
px.scatter(df, width=w, height=h,
    x='param_count_x',  y='secs', log_x=True, log_y=True, color='infer_img_size',
    hover_name='model', hover_data=['infer_samples_per_sec', 'family']
)

## 训练结果


In [ ]:
tdf = get_data('train', 'train_samples_per_sec')

In [ ]:
show_all(tdf, 'Training', 'train_img_size')

In [ ]:
show_subs(tdf, 'Training', 'train_img_size')

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文档已使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言的原始文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或误释，我们概不负责。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
